In [1]:
# !pip install SPARQLWrapper pandas

from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd
import time

In [2]:

# =========================
# 1. Read the missing-name file
# =========================
input_file = "processed/person_missing_name.csv"   # update path if needed
output_file = "processed/person_missing_name_filled.csv"

df_missing = pd.read_csv(input_file)


In [3]:


# Keep only valid QIDs
df_missing["person_id"] = df_missing["person_id"].astype(str).str.strip()
df_missing = df_missing[df_missing["person_id"].str.match(r"^Q\d+$", na=False)].copy()

qids = df_missing["person_id"].drop_duplicates().tolist()

In [7]:
qids = df_missing["person_id"].drop_duplicates().tolist()

print(f"Total QIDs to fetch: {len(qids)}")

Total QIDs to fetch: 926


In [8]:
# =========================
# 2. SPARQL setup
# =========================
sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
sparql.setReturnFormat(JSON)

In [9]:
# =========================
# 3. Helper: query Wikidata labels in batches
# =========================
def fetch_labels_for_qids(qid_batch, max_retries=3, sleep_seconds=2):
    """
    Fetch English labels for a batch of Wikidata QIDs.
    Returns a DataFrame with columns: person_id, wikidata_name
    """
    values_clause = " ".join(f"wd:{qid}" for qid in qid_batch)

    query = f"""
    PREFIX wd: <http://www.wikidata.org/entity/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

    SELECT ?person ?personLabel
    WHERE {{
      VALUES ?person {{ {values_clause} }}

      OPTIONAL {{
        ?person rdfs:label ?personLabel .
        FILTER (LANG(?personLabel) = "en")
      }}
    }}
    """

    for attempt in range(1, max_retries + 1):
        try:
            sparql.setQuery(query)
            results = sparql.query().convert()

            rows = []
            for r in results["results"]["bindings"]:
                person_uri = r["person"]["value"]
                person_id = person_uri.rsplit("/", 1)[-1]
                person_label = r.get("personLabel", {}).get("value")

                rows.append({
                    "person_id": person_id,
                    "wikidata_name": person_label
                })

            return pd.DataFrame(rows)

        except Exception as e:
            print(f"Batch failed (attempt {attempt}/{max_retries}): {e}")
            if attempt < max_retries:
                time.sleep(sleep_seconds)
            else:
                print("Skipping this batch after retries.")
                return pd.DataFrame(columns=["person_id", "wikidata_name"])

In [11]:


# =========================
# 4. Query in batches
# =========================
batch_size = 200   # keep moderate to avoid endpoint issues
all_results = []

for i in range(0, len(qids), batch_size):
    batch = qids[i:i + batch_size]
    print(f"Fetching batch {i//batch_size + 1} / {(len(qids)-1)//batch_size + 1} ...")
    df_batch = fetch_labels_for_qids(batch)
    all_results.append(df_batch)
    time.sleep(1)  # be polite to Wikidata endpoint

df_labels = pd.concat(all_results, ignore_index=True).drop_duplicates(subset=["person_id"])


Fetching batch 1 / 5 ...
Batch failed (attempt 1/3): <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:997)>
Batch failed (attempt 2/3): <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:997)>


KeyboardInterrupt: 

In [ ]:

# =========================
# 5. Merge back to original file
# =========================
df_out = df_missing.merge(df_labels, on="person_id", how="left")

# Replace bad placeholder names (where name == person_id) with Wikidata label
df_out["name_fixed"] = df_out["wikidata_name"]

# If no label found, keep original name
df_out["name_fixed"] = df_out["name_fixed"].fillna(df_out["name"])

# Optional: overwrite the name column
df_out["name"] = df_out["name_fixed"]

# Clean up helper columns if you want
df_out = df_out.drop(columns=["name_fixed"])

# =========================
# 6. Save
# =========================
df_out.to_csv(output_file, index=False)

print(f"Saved: {output_file}")
print(f"Rows in output: {len(df_out)}")
print(f"Names found: {df_out['wikidata_name'].notna().sum()}")
print(f"Names missing after lookup: {df_out['wikidata_name'].isna().sum()}")